<a href="https://colab.research.google.com/github/StrawEater/PracticasPDI3erBimestre/blob/main/Tareas/Agente.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#🤖Agente de Procesamiento de Imagenes

## 🔑 Configuración de API Keys



Para interactuar con los modelos de lenguaje (LLMs) de **Groq** o **Google Gemini**, necesitas definir tus claves de acceso (**API Keys**).

#### 📍 ¿Cómo configurar tus claves?

* **Opción A (Recomendada en Google Colab)**:
  1. Haz clic en el ícono de la **llave de secreto 🔑** en la barra lateral izquierda de Colab.
  2. Crea un nuevo secreto con el nombre `GROQ_API_KEY` o `GOOGLE_API_KEY`.
  3. En **Valor**, pega tu clave correspondiente.
  4. Activa el interruptor para otorgar acceso al notebook.

* **Opción B (Entorno Local o Prueba Rápida)**:
  - Si ejecutas el notebook localmente (VS Code / Jupyter) o quieres probar de forma rápida, puedes pegar tu clave directamente reemplazando `'paste-your-key-here'` en las celdas de código a continuación.

> 🌐 **¿Dónde obtener la API KEY?**
> - **Groq API Key**: [console.groq.com](https://console.groq.com/)
> - **Google Gemini API Key**: [aistudio.google.com](https://aistudio.google.com/)

In [90]:
import os

# Option A: Colab secret (recommended)
try:
  from google.colab import userdata
  os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
  # Option B: paste it directly (fine for a quick test, don't share the notebook after)
  os.environ["GROQ_API_KEY"] = "paste-your-key-here"


In [91]:
import os

# Option A: Colab secret (recommended)
try:
  from google.colab import userdata
  os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
  # Option B: paste it directly (fine for a quick test, don't share the notebook after)
  os.environ["GOOGLE_API_KEY"] ="paste-your-key-here"

Instalar dependecias

In [92]:
!pip install -q mcp groq markdown nest_asyncio ipywidgets google-genai



## Parte 1 — Definición de herramientas

###Guía para Definir tus Herramientas (Tools)

Una **Tool** es una función en Python que puede usar el Agente. Le permite realizar operaciones que los modelos de lenguaje no pueden hacer por sí solos (como procesar imágenes, consultar archivos locales o realizar cálculos precisos).

### 📌 Reglas clave para crear una herramienta:

1. **`@mcp.tool()`**:
   - Se coloca justo encima de la función. Le indica al servidor MCP que registre esta función para que el Modelo pueda verla y llamarla.

2. **Definición Asíncrona (`async def`)**:
   - Todas las herramientas deben definirse con `async def` en lugar de `def`. Esto permite que el agente ejecute llamadas sin bloquear el hilo principal mientras espera entrada/salida (I/O).

3. **Tipado Estricto de Parámetros (`Type Hints`)**:
   - Debes especificar el tipo de cada parámetro (ej: `text: str`, `a: float`) y el tipo de retorno (ej: `-> int`). MCP usa esta información para generar el esquema JSON automáticamente que recibe el LLM.

4. **Instrucción de Uso Clara (`"""..."""`)**:
   - El docstring es la **'instrucción de uso'** que lee el modelo para decidir cuándo usar tu herramienta y qué poner en cada parámetro. Sé conciso y específico.

---
### 💡 Ejemplo de Estructura:
```python
@mcp.tool()
async def mi_herramienta(parametro: str) -> str:
    """Explicación clara de qué hace la herramienta y qué se espera en parametro."""
    # Tu código de procesamiento aquí
    return resultado
```


In [93]:
from mcp.server import MCPServer
import os
import numpy as np
import skimage as ski
import cv2
import nest_asyncio
nest_asyncio.apply()
from pathlib import Path
from skimage import data, io
import uuid
import inspect

mcp = MCPServer("student-demo-server")


### Tools de Test

In [94]:
@mcp.tool()
async def add(a: float, b: float) -> float:
  """Suma dos numeros entre si."""
  return a + b

@mcp.tool()
async def word_count(text: str) -> int:
  """Cuenta la cantidad de palabras en una pieza de texto."""
  return len(text.split())

@mcp.tool()
async def reverse_text(text: str) -> str:
  """Revierte un string."""
  return text[::-1]

### Tools de OS

In [112]:
@mcp.tool()
async def ls(path: str = ".") -> list[dict]:
  """Lista el contenido de un directorio: nombre, tipo (archivo/carpeta) y tamaño en bytes."""
  entradas = sorted(os.listdir(path))
  resultado = []
  for nombre in entradas:
    ruta_completa = os.path.join(path, nombre)
    es_carpeta = os.path.isdir(ruta_completa)
    resultado.append({
      "nombre": nombre,
      "tipo": "carpeta" if es_carpeta else "archivo",
      "tamaño_bytes": None if es_carpeta else os.path.getsize(ruta_completa),
    })
  return resultado

@mcp.tool()
async def copiar_archivo(origen: str, destino: str) -> dict:
  """Copia un archivo a otra carpeta o ruta.

  origen: ruta del archivo a copiar (debe existir).
  destino: carpeta de destino, o ruta completa del archivo de destino.
           Si es una carpeta (existente o terminada en '/'), se conserva
           el nombre original del archivo.
  """
  origen_path = Path(origen)
  if not origen_path.is_file():
    raise FileNotFoundError(f"No se encontró el archivo de origen: {origen}")

  destino_path = Path(destino)
  if destino_path.is_dir() or destino.endswith(("/", os.sep)):
    destino_path = destino_path / origen_path.name

  destino_path.parent.mkdir(parents=True, exist_ok=True)
  shutil.copy2(origen_path, destino_path)

  return {
    "result": f"El archivo se copió correctamente a {destino_path}",
    "name": destino_path.name,
  }

@mcp.tool()
async def eliminar_archivo(path: str) -> dict:
  """Elimina un archivo.

  path: ruta del archivo a eliminar (debe existir y ser un archivo, no una carpeta).
  """
  archivo_path = Path(path)
  if not archivo_path.is_file():
    raise FileNotFoundError(f"No se encontró el archivo: {path}")

  archivo_path.unlink()

  return {
    "result": f"El archivo {path} se eliminó correctamente",
  }

@mcp.tool()
async def leer_txt(path: str, max_caracteres: int = 5000) -> dict:
  """Lee el contenido de un archivo de texto (.txt).

  path: ruta del archivo a leer.
  max_caracteres: cantidad máxima de caracteres a devolver en la respuesta
                  (para no saturar el contexto con archivos muy largos).
                  Por defecto 5000.
  """
  archivo_path = Path(path)
  if not archivo_path.is_file():
    raise FileNotFoundError(f"No se encontró el archivo: {path}")

  contenido = archivo_path.read_text(encoding="utf-8")
  truncado = len(contenido) > max_caracteres

  return {
    "result": contenido[:max_caracteres],
    "truncado": truncado,
    "longitud_total_caracteres": len(contenido),
  }

### Tools de imagenes

#### Utils

In [96]:
# Imágenes disponibles de skimage.data
IMAGENES_SKIMAGE = {
  "astronaut": data.astronaut,
  "brick": data.brick,
  "camera": data.camera,
  "cat": data.chelsea,
  "cell": data.cell,
  "chelsea": data.chelsea,
  "clock": data.clock,
  "coffee": data.coffee,
  "coins": data.coins,
  "colorwheel": data.colorwheel,
  "eagle": data.eagle,
  "grass": data.grass,
  "gravel": data.gravel,
  "horse": data.horse,
  "hubble_deep_field": data.hubble_deep_field,
  "immunohistochemistry": data.immunohistochemistry,
  "kidney": data.kidney,
  "logo": data.logo,
  "microaneurysms": data.microaneurysms,
  "moon": data.moon,
  "page": data.page,
  "retina": data.retina,
  "rocket": data.rocket,
  "shepp_logan_phantom": data.shepp_logan_phantom,
  "text": data.text,
}

def get_scikit_image_by_name(nombre_skimage: str) -> np.ndarray:
  if nombre_skimage is None:
    raise ValueError(
      "Para tipo='skimage' tenés que indicar "
      "nombre_skimage. "
      f"Opciones disponibles: {sorted(IMAGENES_SKIMAGE.keys())}"
    )

  if nombre_skimage not in IMAGENES_SKIMAGE:
    raise ValueError(
      f"Imagen skimage no válida: {nombre_skimage!r}. "
      f"Opciones disponibles: "
      f"{sorted(IMAGENES_SKIMAGE.keys())}"
    )

  # Ejecutamos la función correspondiente
  return IMAGENES_SKIMAGE[nombre_skimage]()


In [97]:
def is_gray_scale(image: np.ndarray) -> bool:
  return image.ndim == 2

def are_color_channel_actually_gray_scale(image: np.ndarray) -> bool:
  red = image[..., 0]
  green = image[..., 1]
  blue = image[..., 2]
  return np.all(red == green) and np.all(red == blue)

def cargar_imagen(path: str) -> np.ndarray:
  """Carga una imagen desde un path y la devuelve como array RGB."""
  imagen = ski.io.imread(path)
  if imagen is None:
    raise ValueError(f"No se pudo cargar la imagen: {path}")

  if is_gray_scale(imagen):
    return imagen

  imagen = imagen[..., :3]

  if are_color_channel_actually_gray_scale(imagen):
    return imagen[..., :0]

  return imagen

In [98]:
def get_iluminance_representation_of_color_image(image):
  ycbcr = ski.color.rgb2ycbcr(image)
  Y, Cb, Cr = ycbcr[..., 0], ycbcr[..., 1], ycbcr[..., 2]
  Y_norm = ((Y - 16) / (235 - 16)) * 255  # rescale Y to [0, 1]
  return Y_norm, Cb, Cr

def reconstruct_image_from_iluminance_representation(Y_norm, Cb, Cr):
  Y = (Y_norm / 255) * (235 - 16) + 16  # back to YCbCr scale
  imagen = ski.color.ycbcr2rgb(
    np.stack([Y, Cb, Cr], axis=-1)
  )
  return np.clip(imagen, 0, 1)

def llamar_funcion_en_imagen(imagen: np.ndarray, funcion):
  """Aplica `funcion` sobre el canal de luminancia (Y) si la imagen es a
    color, o directamente sobre la imagen si ya es en escala de grises.
    Devuelve el resultado crudo de `funcion` (no reconstruye la imagen a
    color ni convierte a uint8)."""
  if not is_gray_scale(imagen):
    Y_norm, _ , _ = get_iluminance_representation_of_color_image(imagen)
    resultado = funcion(Y_norm)
  else:
    resultado = funcion(imagen)

  return resultado

def aplicar_transformacion_a_imagen(imagen: np.ndarray, transformacion):
  """Aplica `transformacion` sobre el canal de luminancia (Y) si la imagen
  es a color, o directamente sobre la imagen si ya es en escala de grises.
  SIEMPRE DEVUELVE uint8"""
  if not is_gray_scale(imagen):
    Y_norm, Cb, Cr = get_iluminance_representation_of_color_image(imagen)
    Y_transformada = transformacion(Y_norm)
    Y_norm = exposure.rescale_intensity(Y_transformada, out_range=(0, 255))
    resultado = reconstruct_image_from_iluminance_representation(Y_norm, Cb, Cr)
  else:
    resultado = transformacion(imagen)

  resultado = ski.util.img_as_ubyte(resultado)
  return resultado


In [99]:
def guardar_imagen(imagen: np.ndarray, path: str):
  """Guarda una imagen en un path."""
  # Crear directorio si no existe
  Path(path).parent.mkdir(
      parents=True,
      exist_ok=True,
  )
  # Guardar imagen
  io.imsave(
      path,
      imagen,
      check_contrast=False,
  )

#### Tools

In [100]:

# Si un dictionario devuelto tiene la key 'show_image_to_user',
# se muestra la imagen del path en la conversacion.
@mcp.tool()
async def mostrar_imagen(path: str) -> dict:
  """Muestra al usuario la imagen de la ruta indicada.

  path: ruta de la imagen a mostrar, incluyendo la extensión (ej. 'salida.png').
  """
  file_path = Path(path)
  if file_path.is_file():
      return {
        "result": f"La imagen en {path} se mostro al usuario",
        "show_image_to_user": path,
      }
  else:
      raise FileNotFoundError(f"No se encontró el archivo: {path}")

@mcp.tool()
async def listar_imagenes_skimage() -> list[str]:
    """
    Lista las imágenes disponibles de skimage.data.
    """
    return sorted(IMAGENES_SKIMAGE.keys())

@mcp.tool()
async def generar_imagen(
    path: str,
    tipo: str = "random",
    nombre_scikit_image: str | None = None,
) -> dict:
  """
  Crea una imagen y la guarda en la ruta indicada.

  path:
    Ruta del archivo de salida, incluyendo extensión.
    Ejemplo: 'salida.png'

  tipo:
    'random': genera ruido gaussiano.
    'skimage': utiliza una imagen de skimage.data.
    'black': genera una imagen negra.

  nombre_scikit_image:
    Nombre de la imagen de skimage.data cuando
    tipo='skimage'.
    Ejemplo: 'astronaut', 'coffee', 'chelsea'.
  """

  if tipo == "random":
    new_image = ski.util.random_noise(
      np.ones((512, 512)) * 0.5,
      mode="gaussian",
      mean=0,
      var=0.05,
    )

  elif tipo == "skimage":
    new_image = get_scikit_image_by_name(nombre_scikit_image)

  elif tipo == "black":
    new_image = np.zeros(
      (512, 512),
      dtype=np.uint8,
    )

  else:
    raise ValueError(
      f"Tipo de imagen no soportado: {tipo!r}. "
      "Usá 'random', 'skimage' o 'black'."
    )

  new_image = ski.util.img_as_ubyte(new_image)
  guardar_imagen(new_image, path)

  return {
    "result": f"La imagen se guardó correctamente en {path}",
    "name": Path(path).name,
  }

## ***Consigna 1)*** Implementación de las Tools del Agente

Completen las *tools* provistas para que el agente cuente con las siguientes funcionalidades:

### - Cálculo de métricas de una imagen

- **Promedio** de intensidad
- **Varianza**
- **Percentil 25**
- **Percentil 75**
- **Métrica del paper** *(la definida en la bibliografía de la materia)*


### - Detección de problemas de una imagen

El agente debe poder enumerar posibles problemas, entre ellos:

- Imagen **oscura**
- Imagen **sobreexpuesta**
- Imagen con **bajo contraste**

### - Corrección de imágenes

- **Aumentar o disminuir el brillo** de una imagen proporcionalmente a un Alpha
- **Mejorar el contraste** de una imagen



In [101]:
from skimage import data, exposure

@mcp.tool()
async def calcular_metricas_imagen(path: str) -> dict:
  """TOOL AUN NO IMPLEMENTADA"""

  return {}

@mcp.tool()
async def detectar_problemas_en_imagen(path: str) -> dict:
  """TOOL AUN NO IMPLEMENTADA"""

  return {}

@mcp.tool()
async def modificar_intensidad_imagen(path: str, alpha: float, path_out: str) -> dict:
  """TOOL AUN NO IMPLEMENTADA"""

  return {}

@mcp.tool()
async def mejorar_contraste_imagen(path: str, path_out: str) -> dict:
  """TOOL AUN NO IMPLEMENTADA"""

  return {}

## Part 2 — Configuración del Agente

Este es todo el codigo para que el Agente funcione, pueden verlo pero no es necesario.

### Render
Renderiza html dentro de colab para poder mostrar mejor los datos

In [102]:

import json
import html as html_lib

import markdown as md_lib
from IPython.display import display, HTML

MD_EXTENSIONS = ["extra", "sane_lists", "nl2br"]


class Rendered:
  """Helpers para mostrar cajas de texto con estilo dentro de un notebook."""

  _BOX_CSS = """
    border-left: 4px solid {color};
    background: {color}15;
    padding: 10px 16px;
    margin: 8px 0;
    border-radius: 6px;
    font-family: -apple-system, "Segoe UI", sans-serif;
    font-size: 13.5px;
    color: #222;
    line-height: 1.5;
  """

  _TABLE_CSS = """
  <style>
    .rendered-md table { border-collapse: collapse; width: 100%; margin: 6px 0; font-size: 13px; }
    .rendered-md th, .rendered-md td { border: 1px solid rgba(0,0,0,0.15); padding: 4px 10px; text-align: left; }
    .rendered-md th { background: rgba(0,0,0,0.05); }
  </style>
  """

  @staticmethod
  def _normalizar_tablas(text: str) -> str:
    """Python-Markdown solo reconoce una tabla si la fila de encabezado
    está en su propio bloque (separado por línea en blanco de cualquier
    texto anterior o posterior). Los modelos no siempre lo respetan,
    así que forzamos esas líneas en blanco antes de renderizar."""
    lineas = (text or "").split("\n")
    resultado = []
    dentro_de_tabla = False

    for linea in lineas:
      es_fila_de_tabla = linea.strip().startswith("|")

      if es_fila_de_tabla and not dentro_de_tabla and resultado and resultado[-1].strip():
        resultado.append("")  # blank line antes de entrar a la tabla
      if dentro_de_tabla and not es_fila_de_tabla and linea.strip():
        resultado.append("")  # blank line al salir de la tabla

      dentro_de_tabla = es_fila_de_tabla
      resultado.append(linea)

    return "\n".join(resultado)

  @staticmethod
  def _markdown_to_html(text: str) -> str:
    texto_normalizado = Rendered._normalizar_tablas(text)
    texto_escapado = html_lib.escape(texto_normalizado or "", quote=False)
    return md_lib.markdown(texto_escapado, extensions=MD_EXTENSIONS)

  @classmethod
  def _wrapper_css(cls, color: str) -> str:
    return cls._BOX_CSS.format(color=color)

  @staticmethod
  def render_box(titulo, contenido_md, color="#6c757d", icono="💬") -> None:
    cuerpo = Rendered._markdown_to_html(contenido_md)
    display(HTML(f"""
    {Rendered._TABLE_CSS}
    <div style="{Rendered._wrapper_css(color)}">
      <div style="font-weight:600; color:{color}; margin-bottom:6px;">{icono} {titulo}</div>
      <div class="rendered-md">{cuerpo}</div>
    </div>
    """))

  @staticmethod
  def render_code_box(titulo: str, contenido, color: str = "#2980b9", icono: str = "🔧") -> None:
    if isinstance(contenido, (dict, list)):
      texto = json.dumps(contenido, ensure_ascii=False, indent=2)
    else:
      texto = str(contenido)
    texto_escapado = html_lib.escape(texto)
    display(HTML(f"""
    <div style="{Rendered._wrapper_css(color)}">
      <div style="font-weight:600; color:{color}; margin-bottom:6px;">{icono} {titulo}</div>
      <pre style="margin:0; white-space:pre-wrap; font-family:Consolas,Monaco,monospace; font-size:12.5px;">{texto_escapado}</pre>
    </div>
    """))

  @staticmethod
  def render_retractable_box(titulo: str, contenido_md: str, color: str = "#8e44ad", icono: str = "🧠") -> None:
    cuerpo = Rendered._markdown_to_html(contenido_md)
    display(HTML(f"""
    <details style="{Rendered._wrapper_css(color)}">
      <summary style="font-weight:600; color:{color}; cursor:pointer;">{icono} {titulo}</summary>
      <div style="margin-top:8px;">{cuerpo}</div>
    </details>
    """))

### Definicion Clases


#### Chatter y Conversacion

In [103]:
from __future__ import annotations

from enum import IntEnum
from typing import Any


class Chatter:
  """Mixin para cualquier emisor de mensajes que se agregan a una Conversacion."""

  def registrar_respuesta(self, conversacion: Conversacion, respuesta) -> None:
    conversacion.agregar_respuesta(self, respuesta)

  def escribir_respuesta(self, respuesta) -> dict:
    """Cada subclase define el formato de su propio mensaje."""
    raise NotImplementedError

  def renderizar_respuesta(self, respuesta, rendering_option: Conversacion.RenderingOptions) -> None:
    """Cada subclase define el formato de su propio renderizado."""
    raise NotImplementedError

  def write_gemini_input(self, respuesta) -> Any:
    """Cada subclase define el formato de su propio input para la Interactions API de Gemini."""
    raise NotImplementedError


class Conversacion:

  class RenderingOptions(IntEnum):
    HIDDEN = 1
    CONVERSATION = 2
    ALL = 3

  def __init__(self, system_prompt: str, rendering_option: RenderingOptions = RenderingOptions.CONVERSATION):
    self.mensajes: list[dict] = [
      {"role": "system", "content": system_prompt},
    ]
    self.system_prompt = system_prompt
    self.historial: list[tuple[Chatter, Any]] = [(None, system_prompt)]
    self.rendering_option = rendering_option

  def cambiar_rendering_option(self, rendering_option: RenderingOptions) -> None:
    self.rendering_option = rendering_option

  def agregar_respuesta(self, mensajero: Chatter, respuesta) -> None:
    mensajero.renderizar_respuesta(respuesta, self.rendering_option)
    self.mensajes.append(mensajero.escribir_respuesta(respuesta))
    # aliasing, pero por ahora no hay drama
    self.historial.append((mensajero, respuesta))

  def replay(self) -> None:
    for mensajero, respuesta in self.historial:
      if mensajero is not None:
        mensajero.renderizar_respuesta(respuesta, self.rendering_option)


class Conversacion_Gemini(Conversacion):
  def __init__(self, system_prompt: str, rendering_option: Conversacion.RenderingOptions = Conversacion.RenderingOptions.CONVERSATION):
    super().__init__(system_prompt, rendering_option)
    self.last_interaction_id = None
    # Lista (no un único valor) porque una misma vuelta puede generar más de
    # un tool call, y cada resultado tiene que viajar en el próximo request.
    self.input_pendiente: list = []

  def agregar_respuesta(self, mensajero: Chatter, respuesta) -> None:
    super().agregar_respuesta(mensajero, respuesta)
    item = mensajero.write_gemini_input(respuesta)
    if item is not None:
      self.input_pendiente.append(item)

#### Proveedor

In [104]:
from __future__ import annotations

import json
from dataclasses import dataclass, field

from groq import Groq
from google import genai
from google.genai import interactions as gint


@dataclass
class RespuestaAgente:
  content: str = ""
  tool_calls: list[_ToolCallUnificado] = field(default_factory=list)
  reasoning: str | None = None


class Proveedor:
  """Interfaz común para los distintos backends de modelos (Groq, Gemini, etc.)."""

  def tool_schema(self, tool) -> dict:
    raise NotImplementedError

  def construir_respuesta_agente(self, respuesta) -> RespuestaAgente:
    raise NotImplementedError

  def _completar(self, modelo_id: str, conversacion: Conversacion, tool_schemas: list | None):
    raise NotImplementedError

  def completar(self, modelo_id: str, conversacion: Conversacion, herramientas: list) -> RespuestaAgente:
    esquemas = [self.tool_schema(h) for h in herramientas] if herramientas else None
    respuesta = self._completar(modelo_id, conversacion, esquemas)
    return self.construir_respuesta_agente(respuesta)

  def iniciar_conversacion(
      self,
      system_prompt: str,
      rendering_option: Conversacion.RenderingOptions = Conversacion.RenderingOptions.CONVERSATION,
  ) -> Conversacion:
    raise NotImplementedError


class _ProveedorGroq(Proveedor):

  def __init__(self, cliente: Groq | None = None):
    self._cliente = cliente or Groq()

  def tool_schema(self, tool) -> dict:
    return {
      "type": "function",
      "function": {
        "name": tool.name,
        "description": tool.description or "",
        "parameters": tool.input_schema,
      },
    }

  def construir_respuesta_agente(self, respuesta) -> RespuestaAgente:
    mensaje = respuesta.choices[0].message

    return RespuestaAgente(
      content=mensaje.content or "",
      tool_calls=[
        _ToolCallUnificado(
          id=tc.id,
          function=_FuncionLlamada(name=tc.function.name, arguments=tc.function.arguments),
        )
        for tc in (mensaje.tool_calls or [])
      ],
      reasoning=getattr(mensaje, "reasoning", None),
    )

  def _completar(self, modelo_id: str, conversacion: Conversacion, tool_schemas: list | None):
    return self._cliente.chat.completions.create(
      model=modelo_id,
      messages=conversacion.mensajes,
      tools=tool_schemas or None,
      tool_choice="auto" if tool_schemas else None,
    )

  def iniciar_conversacion(
      self,
      system_prompt: str,
      rendering_option: Conversacion.RenderingOptions = Conversacion.RenderingOptions.CONVERSATION,
  ) -> Conversacion:
    return Conversacion(system_prompt, rendering_option)


class _ProveedorGemini(Proveedor):

  def __init__(self, cliente: genai.Client | None = None):
    self._cliente = cliente or genai.Client()

  def tool_schema(self, tool) -> dict:
    # ya no hace falta envolver en Tool(function_declarations=[...]),
    # la Interactions API toma una lista plana de declaraciones
    return {
      "type": "function",
      "name": tool.name,
      "description": tool.description or "",
      "parameters": tool.input_schema,
    }

  def _completar(self, modelo_id: str, conversacion: Conversacion_Gemini, tool_schemas: list | None):
    if not conversacion.last_interaction_id:
      interaccion = self._cliente.interactions.create(
        model=modelo_id,
        input=conversacion.input_pendiente,
        tools=tool_schemas or None,
        system_instruction=conversacion.system_prompt,
        generation_config=gint.GenerationConfig(thinking_summaries="auto"),
        stream=False,
      )
    else:
      interaccion = self._cliente.interactions.create(
        model=modelo_id,
        input=conversacion.input_pendiente,
        tools=tool_schemas or None,
        previous_interaction_id=conversacion.last_interaction_id,
      )

    conversacion.last_interaction_id = interaccion.id
    conversacion.input_pendiente = []  # ya se envió, arrancamos de cero para la próxima
    return interaccion

  def construir_respuesta_agente(self, interaccion) -> RespuestaAgente:
    steps = interaccion.steps or []

    tool_calls = [
      _ToolCallUnificado(
        id=s.id,
        function=_FuncionLlamada(name=s.name, arguments=json.dumps(s.arguments or {})),
      )
      for s in steps if s.type == "function_call"
    ]

    # TODO: las `firmas` (thought signatures) se calculan pero no se usan todavía.
    # Si en algún momento necesitás mantener el hilo de "thinking" en llamadas con
    # tool calls multi-turno, esto tendría que viajar de vuelta en el próximo
    # request (habría que agregarle un campo a RespuestaAgente y a write_gemini_input).
    firmas = [s.signature for s in steps if s.type == "thought" and s.signature]

    resumen = "\n".join(
      c.text
      for s in steps if s.type == "thought"
      for c in (s.summary or [])
      if getattr(c, "text", None)
    ) or None

    return RespuestaAgente(
      content=interaccion.output_text or "",
      tool_calls=tool_calls,
      reasoning=resumen,
    )

  def iniciar_conversacion(
      self,
      system_prompt: str,
      rendering_option: Conversacion.RenderingOptions = Conversacion.RenderingOptions.CONVERSATION,
  ) -> Conversacion_Gemini:
    return Conversacion_Gemini(system_prompt, rendering_option)

#### ToolCall

In [105]:
from __future__ import annotations

import json
from dataclasses import dataclass
from IPython.display import Image, display



@dataclass
class _FuncionLlamada:
  name: str
  arguments: str


@dataclass
class _ToolCallUnificado:
  id: str
  function: _FuncionLlamada


class ToolCall(Chatter):
  """Una única invocación de tool call pedida por el modelo."""

  def __init__(self, tool_call: _ToolCallUnificado, mcp: MCPServer):
    self.tool_call = tool_call
    self.tool_name = tool_call.function.name
    self.tool_args = json.loads(tool_call.function.arguments or "{}")
    self.id = tool_call.id
    self.mcp = mcp
    self.es_error = False  # lo termina de setear _ejecutar()

  @staticmethod
  def _is_json(my_str):
    try:
      json.loads(my_str)
      return True
    except ValueError:
      return False

  async def _ejecutar(self) -> str:

    try:
      result = await self.mcp.call_tool(self.tool_name, self.tool_args)
    except Exception as e:
      # Falla de protocolo/transporte (tool inexistente, servidor caído, etc.),
      # no un error de negocio dentro de la tool — igual la tratamos como
      # "es_error" para no tirar abajo toda la conversación por una sola
      # llamada fallida.
      self.es_error = True
      return f"{type(e).__name__}: {e}"

    partes = [item.text for item in result.content if item.type == "text"]

    if not partes:
      return ""

    if len(partes) == 1:
      return partes[0]

    # múltiples bloques: intentamos combinarlos en un array JSON válido
    try:
      objetos = [json.loads(p) for p in partes]
      return json.dumps(objetos, ensure_ascii=False)
    except json.JSONDecodeError:
      return "\n".join(partes)  # no eran JSON, devolvemos texto plano

  async def responder(self, conversacion: Conversacion) -> None:
    respuesta = await self._ejecutar()
    self.registrar_respuesta(conversacion, respuesta)

  def escribir_respuesta(self, respuesta: str) -> dict:
    contenido = f"Error: {respuesta}" if self.es_error else respuesta
    return {
      "role": "tool",
      "tool_call_id": self.id,
      "content": contenido,
    }


  def write_gemini_input(self, respuesta: str) -> dict:
    return {
      "type": "function_result",
      "name": self.tool_name,
      "call_id": self.id,
      "result": [{"type": "text", "text": respuesta}],
      "is_error": self.es_error,
    }

  def renderizar_respuesta(self, respuesta: str, rendering_option: Conversacion.RenderingOptions) -> None:
    if rendering_option >= Conversacion.RenderingOptions.ALL:

      Rendered.render_code_box(f"Llamando herramienta: {self.tool_name}", self.tool_args, color="#2980b9", icono="🔧")

      if self.es_error:
        Rendered.render_code_box(f"Error en {self.tool_name}", respuesta, color="#c0392b", icono="❌")
      else:
        Rendered.render_code_box(f"Resultado: {self.tool_name}", respuesta, color="#27ae60", icono="✅")

    if not self.es_error and rendering_option >= Conversacion.RenderingOptions.CONVERSATION:
      try:
        json_respuesta = json.loads(respuesta)
        if isinstance(json_respuesta, dict) and "show_image_to_user" in json_respuesta:
          display(Image(filename=json_respuesta["show_image_to_user"]))
      except (json.JSONDecodeError, TypeError, ValueError):
        pass

#### Agente

In [106]:
from __future__ import annotations

import textwrap
from enum import Enum


class Agente(Chatter):

  class Modelos(Enum):
    QWEN = ("qwen/qwen3.6-27b", _ProveedorGroq)
    LLAMA = ("llama-3.3-70b-versatile", _ProveedorGroq)
    GEMINI_FLASH = ("gemini-3.6-flash", _ProveedorGemini)
    GEMINI_FLASH_LITE = ("gemini-3.5-flash-lite", _ProveedorGemini)
    GEMINI_PRO = ("gemini-3.6-pro", _ProveedorGemini)

    @property
    def modelo_id(self) -> str:
      return self.value[0]

    @property
    def proveedor(self) -> Proveedor:
      # Cacheamos una única instancia de proveedor por miembro del enum,
      # para no reconstruir el cliente (Groq / genai) en cada llamada.
      if not hasattr(self, "_proveedor_instanciado"):
        proveedor_cls = self.value[1]
        self._proveedor_instanciado = proveedor_cls()
      return self._proveedor_instanciado

  def __init__(
    self,
    nombre: str = "Tron",
    modelo: Modelos = Modelos.QWEN,
    role: str = "You are a helpful AI assistant.",
  ):
    self.nombre = nombre
    self.modelo = modelo
    self.role = role

  def system_prompt(self) -> str:
    return textwrap.dedent(f"""
      You are {self.nombre}.
      {self.role}
    """).strip()

  def escribir_respuesta(self, respuesta) -> dict:
    mensaje = {
      "role": "assistant",
      "content": respuesta.content or "",
    }
    if respuesta.tool_calls:
      mensaje["tool_calls"] = [
        {
          "id": tc.id,
          "type": "function",
          "function": {"name": tc.function.name, "arguments": tc.function.arguments},
        }
        for tc in respuesta.tool_calls
      ]

    reasoning = getattr(respuesta, "reasoning", None)
    if reasoning:
      mensaje["reasoning"] = reasoning

    return mensaje

  def write_gemini_input(self, respuesta) -> None:
    # La respuesta del propio modelo nunca es lo que hay que "reenviar": el
    # próximo input es o bien el resultado de un tool call (ToolCall.write_gemini_input)
    # o bien el próximo mensaje del usuario (Usuario.write_gemini_input). Esto
    # se ignora en agregar_respuesta (ver Conversacion_Gemini) precisamente por eso.
    return None

  def renderizar_respuesta(self, respuesta, rendering_option: Conversacion.RenderingOptions) -> None:
    reasoning = getattr(respuesta, "reasoning", None)
    if rendering_option >= Conversacion.RenderingOptions.ALL and reasoning:
      Rendered.render_retractable_box("Razonamiento interno (click para expandir)", reasoning)

    # "final" solo si esta respuesta no viene acompañada de tool_calls pendientes
    if rendering_option >= Conversacion.RenderingOptions.CONVERSATION and respuesta.content and not respuesta.tool_calls:
      Rendered.render_box("Respuesta final", respuesta.content, color="#c0392b", icono="🎯")

  def responder(self, conversacion: Conversacion, herramientas: list):
    respuesta = self.modelo.proveedor.completar(self.modelo.modelo_id, conversacion, herramientas)
    self.registrar_respuesta(conversacion, respuesta)
    return respuesta

  def iniciar_conversacion(self, rendering_option: Conversacion.RenderingOptions):
    return self.modelo.proveedor.iniciar_conversacion(self.system_prompt(), rendering_option)

  async def continuar_conversacion(self, conversacion: Conversacion, mcp_tools: MCPServer) -> Conversacion:
    herramientas = await mcp_tools.list_tools()

    respuesta = self.responder(conversacion, herramientas)

    while respuesta.tool_calls:
      tool_calls = [ToolCall(tc, mcp_tools) for tc in respuesta.tool_calls]
      for tool_call in tool_calls:
        await tool_call.responder(conversacion)

      respuesta = self.responder(conversacion, herramientas)

    return conversacion

#### Usuario

In [107]:
from __future__ import annotations


class Usuario(Chatter):
  """Representa al usuario humano dentro de la conversación."""

  async def iniciar_conversacion(
      self,
      mensaje_usuario: str,
      agente: Agente,
      mcp_tools: MCPServer,
      rendering_option: Conversacion.RenderingOptions = Conversacion.RenderingOptions.CONVERSATION,
      ) -> Conversacion:

    conversacion = agente.iniciar_conversacion(rendering_option)
    await self.continuar_conversacion(mensaje_usuario, conversacion, agente, mcp_tools)
    return conversacion

  async def continuar_conversacion(
      self,
      mensaje_usuario: str,
      conversacion: Conversacion,
      agente: Agente,
      mcp_tools: MCPServer,
      ) -> Conversacion:

    self.registrar_respuesta(conversacion, mensaje_usuario)
    await agente.continuar_conversacion(conversacion, mcp_tools)
    return conversacion

  def escribir_respuesta(self, respuesta: str) -> dict:
    return {
      "role": "user",
      "content": respuesta,
    }

  def renderizar_respuesta(self, respuesta: str, rendering_option: Conversacion.RenderingOptions) -> None:
    if rendering_option >= Conversacion.RenderingOptions.CONVERSATION:
      Rendered.render_box("Pregunta del usuario", respuesta, color="#34495e", icono="👤")

  def write_gemini_input(self, respuesta):
    # La Interactions API no acepta un string suelto dentro de una lista: cada
    # elemento de `input` tiene que traer su propio campo "type". Un mensaje
    # de usuario es un UserInputStep con contenido de texto adentro.
    return {"type": "user_input", "content": [{"type": "text", "text": respuesta}]}

## Parte 3 -- LLamamos al Agente

### Setup ELEGÍ TU MODELO

In [108]:

agente = Agente(modelo=Agente.Modelos.GEMINI_FLASH_LITE) #Elegí tu modelo entre (QWEN, LLAMA, GEMINI_FLASH, GEMINI_FLASH_LITE, GEMINI_PRO)
usuario = Usuario()
historial_conversaciones = []

In [109]:
async def handle_prompt_submit(texto_input):
  prompt = texto_input.value
  texto_input.value = ""
  conversacion = await usuario.iniciar_conversacion(prompt, agente, mcp, Conversacion.RenderingOptions.ALL)
  historial_conversaciones.append(conversacion)

In [110]:
# @title Probá el Agente
import asyncio
import nest_asyncio
import ipywidgets as widgets
from IPython.display import display

nest_asyncio.apply()  # permite anidar el loop dentro del loop del kernel

texto_input = widgets.Textarea(
    placeholder='Escribí tu pregunta acá (Shift+Enter o click en Enviar)...',
    layout=widgets.Layout(width='100%', height='100px')
)

boton_enviar = widgets.Button(
    description='Enviar',
    button_style='primary',
    icon='paper-plane',
    layout=widgets.Layout(margin='6px 0 12px 0', width='140px')
)

salida = widgets.Output()

def _on_click_enviar(b):
    b.disabled = True
    b.description = 'Pensando...'
    b.icon = 'hourglass-half'
    with salida:
        salida.clear_output()
        try:
            asyncio.run(handle_prompt_submit(texto_input))
        except Exception as e:
            print(f"⚠️ Error: {e}")
        finally:
            b.disabled = False
            b.description = 'Enviar'
            b.icon = 'paper-plane'

boton_enviar.on_click(_on_click_enviar)

ui = widgets.VBox([texto_input, boton_enviar, salida])
display(ui)


### Retomar una conversacion

In [111]:
if len(historial_conversaciones):
  conversacion_anterior = historial_conversaciones[0]
  conversacion_anterior.replay() # Mostramos la conversacion pasada
  await usuario.continuar_conversacion("Podrias desarrollar?", conversacion_anterior, agente, mcp)
  print()

## 2) Demostración de uso de las Tools

Mostrar un **prompt** y la **conversación completa** resultante, en donde se observe al agente utilizando *cada una* de las herramientas implementadas en el punto anterior.

## 3) Automatización del escenario completo

Configurar al agente para que sea capaz de automatizar el siguiente escenario:

> ### 📰 Escenario: Editor de fotos en un diario
>
> Te han contratado en un diario como **editor de fotos**. Tu trabajo consiste en:
>
> 1. **Recibir** noticias en formato `.txt` y leerlas.
> 2. **Buscar**, dentro de la colección de imágenes disponible, cuáles podrían ser útiles para complementar la nota.
> 3. **Subirlas** a una carpeta dentro de `historias_a_validar` con el mismo nombre que la nota y con ella misma, para que el jefe editorial las apruebe.
>
> **Importante:** algunas fotografías del banco de imágenes pueden no ser de la mejor calidad, por ejemplo, estar muy oscuras o carecer de contraste. En esos casos, **es necesario mejorarlas antes de subirlas**, o de lo contrario serán **rechazadas inmediatamente**.